# Convert STS EPD to Strategy CSV

Run this utility before `03a_build_strategy_datasets.ipynb`. It converts an STS-Rating `.epd` file into the CSV schema consumed by `configs/strategy.yaml -> strategy_dataset.sources`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project Paths

Place the EPD file at `Chess/data/strategy_sources/sts/STS1-STS5_LAN_v6.epd`. The converted CSV is written beside the other strategy sources.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")

sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(PROJECT_ROOT / "requirements_colab.txt"),
])

EPD_PATH = PROJECT_ROOT / "data/strategy_sources/sts/STS1-STS5_LAN_v6.epd"
CSV_PATH = PROJECT_ROOT / "data/strategy_sources/sts_rating_converted.csv"

print("Input:", EPD_PATH)
print("Output:", CSV_PATH)

## Convert

The STS `bm` opcode becomes `best_move`, `policy_target`, and `acceptable_moves`. Rows are marked as `annotated_benchmark_games / user_provided` and `split: test`, so they are evaluation material rather than ordinary training data.

In [ ]:
import csv
import json
from datetime import datetime, timezone
import chess

if not EPD_PATH.is_file():
    raise FileNotFoundError(f"Put STS1-STS5_LAN_v6.epd here first: {EPD_PATH}")

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
rows = []
skipped_without_bm = 0

for line_number, line in enumerate(
    EPD_PATH.read_text(encoding="utf-8", errors="ignore").splitlines(), 1
):
    line = line.strip()
    if not line or line.startswith("#"):
        continue

    board = chess.Board()
    operations = board.set_epd(line)
    best = operations.get("bm", [])
    if isinstance(best, chess.Move):
        best = [best]
    best_uci = [move.uci() for move in best]
    if not best_uci:
        skipped_without_bm += 1
        continue

    probability = 1.0 / len(best_uci)
    source_id = str(operations.get("id") or f"sts-v6-{line_number:05d}")
    comment = str(operations.get("c0") or "")

    rows.append(
        {
            "fen": board.fen(),
            "best_move": best_uci[0],
            "policy_target": json.dumps({move: probability for move in best_uci}),
            "acceptable_moves": json.dumps(best_uci),
            "theme": "annotated_benchmark_games",
            "subtheme": "user_provided",
            "source": "STS-Rating STS1-STS5_LAN_v6",
            "source_id": source_id,
            "source_game_id": source_id,
            "source_line_id": str(line_number),
            "split": "test",
            "created_at": datetime.now(timezone.utc).isoformat(),
            "label_kind": "user_supplied",
            "label_provenance": json.dumps(
                {"kind": "sts_epd_bm", "opcode": "bm", "source_file": EPD_PATH.name}
            ),
            "notes": comment,
        }
    )

if not rows:
    raise ValueError("No STS rows with bm best-move opcodes were found.")

fieldnames = sorted({key for row in rows for key in row})
with CSV_PATH.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("Wrote", len(rows), "rows to", CSV_PATH)
print("Skipped rows without bm:", skipped_without_bm)

## Validate Preview

This loads a few converted rows through the repo's strategy schema.

In [ ]:
from chess_rl.strategy_dataset import load_source

preview_spec = {"path": "data/strategy_sources/sts_rating_converted.csv"}
preview = []
for index, row in enumerate(load_source(PROJECT_ROOT, preview_spec)):
    preview.append({"fen": row["fen"], "best_move": row["best_move"], "source_id": row["source_id"]})
    if index == 4:
        break

preview